In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from networkx.algorithms import community
import math
import warnings
import os

warnings.filterwarnings('ignore')

# ================= 配置区 =================
# 读取带有类型的详细图谱数据 (已修改为PPMI的图谱文件)
FULL_KG_PATH = "PPMI_DementiaHKG_PrimeKG_Sub.csv"  
OUTPUT_PDF_PATH = "PPMI-DementiaHKGE.pdf"

def plot_static_kg_pdf_with_stats():
    print("==================================================")
    print("📊 开始统计 DementiaHKG 图谱数据并绘制静态高清 PDF...")
    
    if not os.path.exists(FULL_KG_PATH):
        print(f"❌ 找不到全量图谱文件: {FULL_KG_PATH}，请确保路径正确。")
        return
        
    df = pd.read_csv(FULL_KG_PATH)
    
    # ---------------- 1. 数据统计 ----------------
    # 提取三元组总数
    num_triplets = len(df)
    
    # 提取所有去重后的节点及其类型
    nodes_x = df[['x_name', 'x_type']].rename(columns={'x_name': 'node', 'x_type': 'type'})
    nodes_y = df[['y_name', 'y_type']].rename(columns={'y_name': 'node', 'y_type': 'type'})
    all_nodes_df = pd.concat([nodes_x, nodes_y]).drop_duplicates(subset=['node'])
    
    num_nodes = len(all_nodes_df)
    
    # 构建有向图以获取真实的图谱边数
    G = nx.from_pandas_edgelist(df, source='x_name', target='y_name', edge_attr='relation', create_using=nx.DiGraph())
    num_edges = G.number_of_edges()
    
    # 统计每种节点类型的数量
    node_type_counts = all_nodes_df['type'].value_counts()
    
    # 统计每种关系类型的数量
    relation_counts = df['relation'].value_counts()
    
    # ---------------- 2. 控制台打印 ----------------
    print(f"\n📈 [整体规模统计]:")
    print(f"   - 节点总数: {num_nodes}")
    print(f"   - 边总数: {num_edges}")
    print(f"   - 三元组总数: {num_triplets}")
    
    print(f"\n🧬 [各类节点数量统计]:")
    for ntype, count in node_type_counts.items():
        print(f"   - {ntype}: {count}")
        
    print(f"\n🔗 [各类关系数量统计]:")
    for rel, count in relation_counts.items():
        print(f"   - {rel}: {count}")
        
    # ---------------- 3. 图谱绘制 (脉络化社区布局) ----------------
    print("\n⏳ 正在进行脉络化社区布局计算 (节点较多，可能需要 1-2 分钟，请耐心等待)...")
    
    # 记录每个节点的类型用于映射颜色
    node_types = dict(zip(all_nodes_df['node'], all_nodes_df['type']))
        
    # 颜色映射字典
    color_map = {
        'disease': '#ff4757',            
        'effect/phenotype': '#ffa502',   
        'gene/protein': '#2ed573',       
        'anatomy': '#1e90ff',            
        'biological_process': '#ff6b81', 
        'pathway': '#9c88ff',            
        'cellular_component': '#7bed9f', 
        'molecular_function': '#70a1ff'  
    }
    
    # 开始渲染 Matplotlib 图像
    plt.figure(figsize=(24, 24), facecolor='#1a1a1a') # 使用深色背景提升对比度
    ax = plt.gca()
    ax.set_facecolor('#1a1a1a')
    
    # 进行社区检测 (使用无向图版本以适配算法)
    G_undirected = G.to_undirected()
    communities_generator = community.greedy_modularity_communities(G_undirected)
    communities = sorted(communities_generator, key=len, reverse=True)
    num_communities = len(communities)
    print(f"   ✅ 发现 {num_communities} 个主要社区岛屿，正在生成脉络...")

    # 分层布局：先将社区中心排布在圆环上，再在局部展开节点
    meta_pos = {}
    radius = 15.0 
    for i in range(num_communities):
        angle = (2 * math.pi * i) / num_communities
        meta_pos[i] = (radius * math.cos(angle), radius * math.sin(angle))

    pos = {}
    for i, comm in enumerate(communities):
        subgraph = G_undirected.subgraph(comm)
        sub_pos = nx.spring_layout(subgraph, k=0.15, iterations=20, seed=2026, center=meta_pos[i], scale=2.5)
        for node, (x, y) in sub_pos.items():
            pos[node] = (x, y)
    
    # 获取严格对齐的节点列表与颜色序列
    nodelist = list(G.nodes())
    node_colors = [color_map.get(node_types.get(node, 'default'), '#cccccc') for node in nodelist]
    
    # 绘制节点和边 (引入曲线脉络效果)
    nx.draw_networkx_nodes(G, pos, nodelist=nodelist, node_color=node_colors, node_size=18, alpha=0.9, edgecolors='none', ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.15, edge_color='#7f8c8d', width=0.1, arrows=False, connectionstyle='arc3,rad=0.05', ax=ax)
    
    # ---------------- 4. 绘制左上角图例 ----------------
    legend_elements = []
    
    # 添加节点类型颜色图例
    for ntype, color in color_map.items():
        if ntype in node_type_counts: # 只显示图谱中实际存在的类型
            legend_elements.append(mpatches.Patch(color=color, label=f"{ntype} ({node_type_counts[ntype]})"))
        
    # 设置图例格式，放置在左上角
    legend = plt.legend(handles=legend_elements, loc='upper left', fontsize=16, 
                        frameon=True, facecolor='#2c3e50', edgecolor='white', labelcolor='white', title="Node Types", title_fontsize=18)
    legend.get_title().set_color("white")
    
    # 修改标题为 PPMI
    plt.title('PPMI-DementiaHKG Knowledge Graph', fontsize=30, color='white', fontweight='bold', pad=20)
    plt.axis('off')
    
    # ---------------- 5. 导出并保存为 PDF ----------------
    print(f"\n💾 正在导出矢量 PDF 文件...")
    plt.savefig(OUTPUT_PDF_PATH, format='pdf', bbox_inches='tight', facecolor='#1a1a1a')
    plt.close()
    
    print(f"   🎉 绘制成功！文件已保存至: {OUTPUT_PDF_PATH}")
    print("==================================================")

if __name__ == "__main__":
    plot_static_kg_pdf_with_stats()

📊 开始统计 DementiaHKG 图谱数据并绘制静态高清 PDF...

📈 [整体规模统计]:
   - 节点总数: 2497
   - 边总数: 13039
   - 三元组总数: 13050

🧬 [各类节点数量统计]:
   - gene/protein: 1287
   - disease: 480
   - biological_process: 296
   - effect/phenotype: 168
   - anatomy: 134
   - cellular_component: 68
   - molecular_function: 60
   - pathway: 4

🔗 [各类关系数量统计]:
   - anatomy_protein_present: 3487
   - disease_phenotype_positive: 2953
   - protein_protein: 1650
   - bioprocess_protein: 1488
   - disease_protein: 1370
   - molfunc_protein: 943
   - cellcomp_protein: 457
   - phenotype_protein: 261
   - bioprocess_bioprocess: 113
   - phenotype_phenotype: 110
   - disease_disease: 92
   - disease_phenotype_negative: 39
   - pathway_protein: 28
   - anatomy_protein_absent: 20
   - molfunc_molfunc: 19
   - cellcomp_cellcomp: 14
   - anatomy_anatomy: 6

⏳ 正在进行脉络化社区布局计算 (节点较多，可能需要 1-2 分钟，请耐心等待)...
   ✅ 发现 97 个主要社区岛屿，正在生成脉络...

💾 正在导出矢量 PDF 文件...
   🎉 绘制成功！文件已保存至: PPMI-DementiaHKGE.pdf
